# Convolutional Neural Networks for Eye Disease Detection

## Authors

Laura Cahill, Olivia Jones-Martin, Roberto Mercado, Zuriel Pagan

## Dataset

We use the [Retinal Fundus Multi-disease Image Dataset (RFMiD)](https://ieee-dataport.org/open-access/retinal-fundus-multi-disease-image-dataset-rfmid?check_logged_in=1) for this project, which is available on the IEEE DataPort website. It contains 3200 fundus color images captured by three different fundus cameras. The dataset is divided into a training set of 1920 images, validation set of 640 images, and testing set of 640 images. Ground truth labels are provided through CSV files, which account for 1 healthy class and 45 disease classes.

## Prerequisites

The following Python 3.12 modules are required to execute the cells of this notebook.

In [2]:
from PIL import Image, ImageEnhance, ImageOps
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import pandas as pd
import os
import re
import seaborn as sns
import shutil

The following constants assume that the "All Classes" version of the dataset was downloaded from the IEEE DataPort website and placed into the same directory as this notebook.

In [3]:
DATA_PATH = 'A. RFMiD_All_Classes_Dataset'
IMAGE_PATH = f'{DATA_PATH}/1. Original Images'
GROUND_TRUTH_PATH = f'{DATA_PATH}/2. Groundtruths'
TRAINING_SET_PATH = f'{IMAGE_PATH}/a. Training Set'
VALIDATION_SET_PATH = f'{IMAGE_PATH}/b. Validation Set'
TESTING_SET_PATH = f'{IMAGE_PATH}/c. Testing Set'

## Data Compression

Given the size of the images within the dataset, we had to compress the images before we uploading them to University of Massachusetts Lowell's GPU servers. To do this, we cropped the black borders surrounding the eye images and shrank the images down to 1028x1028 using the following commented code. This allowed our dataset to fit within our alloted disk space on the GPU server.

### Obtaining Image Paths

In [4]:
# DATASET_DIR = "./A. RFMiD_All_Classes_Dataset"
# image_paths = []

# def listImages(directory: str, image_path_array: list):
#     for subpath in os.listdir(directory):
#         subpath = os.path.join(directory, subpath)
#         print(subpath)
#         if re.search(r'\.(jpg|jpeg|png)$', subpath):
#             image_path_array.append(os.path.join(subpath))
#         if os.path.isdir(os.path.join(subpath)):
#             listImages(subpath, image_path_array)

# listImages(DATASET_DIR, image_paths)
# print(image_paths)

### Obtaining Image Sizes

In [5]:
# image_size_list = []
# image_size_dict = dict()

# for image_path in image_paths:
#     img = Image.open(image_path)
#     image_size_list.append((img.width, img.height))
#     if f"{img.width}-{img.height}" not in image_size_dict:
#         image_size_dict[f"{img.width}-{img.height}"] = []
#     image_size_dict[f"{img.width}-{img.height}"].append(image_path)

# image_size_list = list(set(image_size_list))
# for size in image_size_list:
#     print(size)
#     print(image_size_dict[f"{size[0]}-{size[1]}"])

### Compressing Images

In [6]:
# for path in image_paths:

#     # Open the image
#     img = Image.open(path)

#     width, height = img.size

#     # if it is already cropped, skip it
#     if width == 1028 and height == 1028:
#         img.close()
#         continue

#     # Convert to grayscale to find the light vs dark
#     bw = img.convert('L')

#     # Make comparison image a tenth of the size, in a bilinear way, to blur out any excess pixels
#     bw = bw.resize((width // 10, height // 10), resample=Image.BILINEAR)

#     # bring back to original size
#     bw = bw.resize((width, height), resample=Image.BILINEAR)

#     bw = ImageEnhance.Contrast(bw).enhance(2.0)

#     # getbbox finds the box around the eye
#     eye_box = bw.getbbox()

#     #if eye_box is real:

#     if eye_box:
#         img = img.crop(eye_box)

#     else:
#         print(f"Warning: Could not find eye box for image {path}. Skipping cropping.")

#     # Resize eye box to 1028x1028
#     img = img.resize((1028, 1028))

#     #Save it
#     img.save(path)
#     img.close()

# print("images have been cropped and resized to 1028x1028!")

## Data Extraction

Here, we load the ground truth labels associated with the training, validation, and testing sets into memory.

In [7]:
training_metadata = pd.read_csv(f'{GROUND_TRUTH_PATH}/a. RFMiD_Training_Labels.csv')
validation_metadata = pd.read_csv(f'{GROUND_TRUTH_PATH}/b. RFMiD_Validation_Labels.csv')
testing_metadata = pd.read_csv(f'{GROUND_TRUTH_PATH}/c. RFMiD_Testing_Labels.csv')